# Encrypted Machine Learning: Clustering (`ml/clustering.py`)

This tutorial covers `src/concrete_fhe_toolkit/ml/clustering.py`. This module provides an `sklearn`-style entry point for unsupervised grouping of encrypted samples (like K-Means) and exports clustering metrics such as `inertia`.

## 1. Encrypted Clustering Metrics (`inertia`)

Inertia calculates the sum of squared distances from each encrypted sample to its nearest public centroid. Lower inertia means better clustering.

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.ml.clustering import inertia

def test_inertia(x1: int, y1: int, x2: int, y2: int):
    samples = [[x1, y1], [x2, y2]]
    # Public Centroids at (0,0) and (10,10)
    centroids = [[0, 0], [10, 10]]
    
    # max_distance sizes the internal min() argmin reduction
    return inertia(samples, centroids, max_distance=300)

compiler = fhe.Compiler(test_inertia, {
    "x1": "encrypted", "y1": "encrypted", 
    "x2": "encrypted", "y2": "encrypted"
})

# Input boundaries
inputset = [(1, 1, 9, 9), (0, 0, 10, 10)]
circuit = compiler.compile(inputset)

# Test samples: Point A (1,1) is near (0,0), Point B (9,9) is near (10,10)
# Sq Dist from A to (0,0) = 1^2 + 1^2 = 2
# Sq Dist from B to (10,10) = (-1)^2 + (-1)^2 = 2
# Total inertia = 2 + 2 = 4
calculated_inertia = circuit.encrypt_run_decrypt(1, 1, 9, 9)

assert calculated_inertia == 4
print("✅ Encrypted Inertia calculation passed!")